In [4]:
import pandas as pd
import json
import re
from pathlib import Path
from pdf2image import convert_from_path
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from google.oauth2.credentials import Credentials
from google.genai import errors as genai_errors
import mimetypes


# ==========================
# CONFIGURATION (COMPANY VM)
# ==========================

base_url = "https://vertexai.prod.ai-gateway.quantumblack.com/7a4f3d63-b5db-4d5b-8076-8f114d1f14f7/"
access_token = "eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJhZXNKN2kxNGNidnVuTU40MTJrOU5yZ2ROeENhTlJudTNPbC1TU08ycFlJIn0.eyJleHAiOjE3NjQ5MDc0NzQsImlhdCI6MTc2NDkwNTY3NCwiYXV0aF90aW1lIjoxNzY0OTA1Njc0LCJqdGkiOiIzZDJhOTk4Yy0xNTM3LTQ5ZmUtODczYi0zNjU0ZmVkMjQwMjIiLCJpc3MiOiJodHRwczovL2F1dGgubWNraW5zZXkuaWQvYXV0aC9yZWFsbXMvciIsImF1ZCI6ImJjZDIzNzI4LTNkMjctNDQ3Yy1hMGE5LWVhY2FmMzkzYTZmNSIsInN1YiI6IjI0NDRiYzZjLTAwMzctNGIyZS1hYzI3LWZjNTlhNTkxNTM2NiIsInR5cCI6IklEIiwiYXpwIjoiYmNkMjM3MjgtM2QyNy00NDdjLWEwYTktZWFjYWYzOTNhNmY1Iiwic2Vzc2lvbl9zdGF0ZSI6IjM3ZjI2MTMyLTQyN2UtNDQyMi1iYmRmLTAxMDA0YWNmNGU1NiIsImF0X2hhc2giOiItcDdnTUlOZXJhQlQ5WW9LbElDTVNBIiwibmFtZSI6IlVnYW5kaGFyIFZhZGRpIiwiZ2l2ZW5fbmFtZSI6IlVnYW5kaGFyIiwiZmFtaWx5X25hbWUiOiJWYWRkaSIsInByZWZlcnJlZF91c2VybmFtZSI6IjE1ZDhiYmNkMmMzNTNmYWUiLCJlbWFpbCI6IlVnYW5kaGFyX1ZhZGRpQG1ja2luc2V5LmNvbSIsImFjciI6IjEiLCJzaWQiOiIzN2YyNjEzMi00MjdlLTQ0MjItYmJkZi0wMTAwNGFjZjRlNTYiLCJlbWFpbF92ZXJpZmllZCI6dHJ1ZSwiZm1ubyI6IjM0NzI3NiIsImdyb3VwcyI6WyI3YTRmM2Q2My1iNWRiLTRkNWItODA3Ni04ZjExNGQxZjE0ZjciLCJBbGwgRmlybSBVc2VycyJdfQ.g55gAN9vas_qs5pSgpvKStoxQ9CP4nlXll-2Enb-XrRJ9H5Rhw9gG42XV2vsLOJQHW9guxT5clRwwm13pHaXrdwH4oYNDjZmJ3Z9g9Kgh4zZ7-AdbVCAlr1blReRM7F9bjLnysL4ZeXxVTggd5fLZVxoSURAnWGjX-gYlLxD4azCzp1IR3nIYwvxX44qxtTENvZln2hgmpk_Z3XkBrmclawqas_JvV-DJ6ixKhrk91mS_I3DUzm2YFBwuwiedvahf0KpvTyrxU1M3U2gJvpX5XXqH7p41GSQqu0VE5-B4t3liYLIUxuqAvtkVYeN3Al3lnY1zsEPhvQOBYF0yPwVUA"
credentials = Credentials(access_token)

client = genai.Client(
    http_options=HttpOptions(
        api_version="v1",
        base_url=base_url,
    ),
    vertexai=True,
    project="aigateway",
    location="global",
    credentials=credentials,
)

MODEL_NAME = "gemini-2.5-pro"

print("Enhanced Connection Extractor Ready (Table-Focused with Segment Mapping).")


# ==========================
# ENHANCED PROMPT FOR TABLE-BASED EXTRACTION
# ==========================

def get_enhanced_connection_prompt():
    return """
    You are an Engineering Wiring/Diagram Connection Extraction AI with strong
    2D spatial understanding and table reading capabilities.

    INPUT:
    A technical engineering drawing that contains:
    - Connection tables (wire lists, cable schedules, connector mapping tables)
    - Wiring diagrams with routing paths
    - Dimension annotations showing distances between points
    - Junction points, splices, or intermediate connection points

    YOUR PRIMARY TASK:

    1. LOCATE AND READ CONNECTION TABLES
       First, identify any tables that map connections between components.
       These tables typically have columns like:
       - From Connector / Source / Start Point
       - To Connector / Destination / End Point
       - Wire ID / Cable ID
       - Wire Gauge / Size
       - Color / Color Code
       - Length / Distance

       Common table titles:
       - "Wire List"
       - "Cable Schedule"
       - "Connection Table"
       - "Harness Routing"
       - "Interconnect Matrix"

    2. FOR EACH CONNECTION IN THE TABLE:
       Extract the connector-to-connector mapping:
       
       - from_connector: Source connector ID (e.g., "Connector5", "J5", "P5")
       - to_connector: Destination connector ID (e.g., "Connector6", "J6", "P6")
       - wire_id: Wire/cable identifier
       - wire_gauge: Cross-section or gauge
       - wire_color: Wire color code
       - core_count: Number of cores (if multi-core cable)

    3. TRACE THE PHYSICAL PATH ON THE DIAGRAM:
       For each connection, follow the wire routing on the diagram:
       
       - Start from the FROM connector
       - Follow the line/wire path
       - Identify ALL intermediate points (junctions, splices, bends, terminals)
       - Look for dimension labels along EACH segment between points
       - End at the TO connector

       Create a segment breakdown where EACH segment represents the distance
       between two consecutive points along the path.

    4. SEGMENT LENGTH EXTRACTION:
       For each segment between points, extract the dimension label:
       
       Examples of path breakdown:
       - Connector5 → Junction1: "7 ft"
       - Junction1 → Junction2: "3 ft"
       - Junction2 → Connector6: "2 ft"
       
       Result: segment_lengths = ["7 ft", "3 ft", "2 ft"]

       If a segment has no dimension label, use null for that segment.
       If the entire path has no dimensions, use: segment_lengths = []

    5. IDENTIFY JUNCTION POINTS (OPTIONAL BUT HELPFUL):
       If you can identify junction/splice point names or IDs, include them:
       
       junction_points = ["Junction1", "Junction2", "Splice_A"]
       
       This helps understand the path structure.

    6. OUTPUT FORMAT (STRICT JSON, NO MARKDOWN):
       Return ONLY a valid JSON object in this exact format:

       {
         "connections": [
           {
             "from_connector": "Connector5",
             "to_connector": "Connector6",
             "wire_id": "W101",
             "wire_gauge": "18 AWG",
             "wire_color": "Blue/White",
             "core_count": 1,
             "segment_lengths": ["7 ft", "3 ft", "2 ft"],
             "junction_points": ["Junction1", "Junction2"],
             "total_segments": 3,
             "notes": null
           }
         ],
         "table_found": true,
         "table_location": "Bottom right of drawing"
       }

    CRITICAL RULES:
    - Focus on CONNECTOR-TO-CONNECTOR mappings (not individual pins)
    - Break down the path into ALL segments between intermediate points
    - Each segment represents ONE hop in the routing path
    - Use null for missing data, [] for empty arrays
    - If no table found: return {"connections": [], "table_found": false}
    - Include the "total_segments" count for validation
    - Do NOT calculate total length, only list segment lengths
    - Preserve all units exactly as shown (ft, in, mm, m, etc.)
    - If multiple tables exist, extract from all of them

    IMPORTANT: This is about routing paths with multiple segments/junctions,
    not just direct point-to-point connections. Break down the complete path
    from source connector through all intermediate points to destination connector.
    """


# ==========================
# MODEL CALL: ONE IMAGE → JSON
# ==========================

def analyze_image_enhanced(image_path: str):
    """
    Enhanced analysis focusing on table extraction and segment mapping.
    Returns a dict with structure:
    {
      "connections": [...],
      "table_found": bool,
      "table_location": str
    }
    """
    print(f"   -> Analyzing: {image_path}...")

    with open(image_path, "rb") as f:
        image_bytes = f.read()

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"

    image_part = types.Part.from_bytes(
        data=image_bytes,
        mime_type=mime_type,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[get_enhanced_connection_prompt(), image_part],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
            ),
        )

        raw_text = response.text or "{}"

        # Strip code fences if present
        if "```json" in raw_text:
            raw_text = raw_text.split("```json")[1].split("```")[0]
        elif "```" in raw_text:
            raw_text = raw_text.split("```")[1].split("```")[0]

        data = json.loads(raw_text)

    except genai_errors.ClientError as e:
        print("      [ClientError] Gateway / auth / model issue.")
        print("      Message:", getattr(e, "message", e))
        return {"connections": [], "table_found": False}

    except Exception as e:
        print("      [Error] Could not analyze this page.")
        print("      Detail:", e)
        return {"connections": [], "table_found": False}

    # Normalize structure
    if not isinstance(data, dict):
        data = {"connections": [], "table_found": False}
    if "connections" not in data:
        data["connections"] = []
    if "table_found" not in data:
        data["table_found"] = False

    return data


# ==========================
# MAIN PIPELINE: FILE → EXCEL
# ==========================

def process_file_enhanced(file_path: str):
    """
    Enhanced processing with focus on table-based connector mappings
    and segmented path lengths.
    """
    print(f"\n{'='*60}")
    print(f"Processing Engineering Drawing: {file_path}")
    print(f"{'='*60}")
    
    ext = Path(file_path).suffix.lower()
    temp_images = []

    # 1) PDF → per-page images
    if ext == ".pdf":
        print("\n📄 Converting PDF to images (300 dpi)...")
        try:
            # Check if file exists
            if not Path(file_path).exists():
                print(f"   ❌ ERROR: File not found: {file_path}")
                return
            
            # Check file size
            file_size = Path(file_path).stat().st_size / (1024 * 1024)  # MB
            print(f"   → File size: {file_size:.2f} MB")
            
            # Try conversion with better error handling
            pages = convert_from_path(
                file_path, 
                dpi=300,
                fmt='png',
                thread_count=1
            )
            
            if not pages or len(pages) == 0:
                print(f"   ❌ ERROR: PDF has 0 pages or failed to convert")
                print(f"   → Try opening the PDF manually to verify it's valid")
                return
            
            print(f"   ✓ PDF has {len(pages)} pages")
            
            for i, p in enumerate(pages, start=1):
                img_path = f"temp_conn_page_{i}.png"
                p.save(img_path, "PNG")
                temp_images.append(img_path)
                print(f"   → Saved page {i} as {img_path}")
                
            print(f"   ✓ Converted {len(pages)} pages successfully")
            
        except Exception as e:
            print(f"   ❌ ERROR during PDF conversion: {type(e).__name__}")
            print(f"   → Details: {str(e)}")
            print(f"\n   Troubleshooting:")
            print(f"   1. Verify poppler-utils is installed (required for pdf2image)")
            print(f"   2. Check if PDF is password protected or corrupted")
            print(f"   3. Try: pip install pdf2image pillow")
            print(f"   4. On Linux: sudo apt-get install poppler-utils")
            print(f"   5. On Mac: brew install poppler")
            print(f"   6. On Windows: Download poppler and add to PATH")
            return
    else:
        print(f"\n🖼️ Processing image file directly...")
        if not Path(file_path).exists():
            print(f"   ❌ ERROR: File not found: {file_path}")
            return
        temp_images = [file_path]
        print(f"   ✓ Image file found")

    all_connections = []
    tables_found = []

    # 2) Per-page analysis
    for page_idx, img in enumerate(temp_images, start=1):
        print(f"\n{'─'*60}")
        print(f"📊 Analyzing Page {page_idx}/{len(temp_images)}")
        print(f"{'─'*60}")
        
        page_data = analyze_image_enhanced(img)
        
        table_found = page_data.get("table_found", False)
        table_location = page_data.get("table_location", "N/A")
        conns = page_data.get("connections", [])

        if table_found:
            tables_found.append({
                "page": page_idx,
                "location": table_location
            })
            print(f"   ✓ Table found: {table_location}")
        else:
            print(f"   ⚠ No connection table detected on this page")

        if not isinstance(conns, list):
            print("   ⚠ Unexpected data format, skipping...")
            continue

        print(f"   → Extracted {len(conns)} connections")

        for conn_idx, c in enumerate(conns, start=1):
            from_conn = c.get("from_connector")
            to_conn = c.get("to_connector")
            wire_id = c.get("wire_id")
            wire_gauge = c.get("wire_gauge")
            wire_color = c.get("wire_color")
            core_count = c.get("core_count")
            segments = c.get("segment_lengths", [])
            junctions = c.get("junction_points", [])
            notes = c.get("notes")

            # Normalize core_count
            if core_count is not None and not isinstance(core_count, int):
                try:
                    core_count = int(core_count)
                except Exception:
                    m = re.search(r"\d+", str(core_count))
                    core_count = int(m.group()) if m else None

            # Ensure segments is a list
            if not isinstance(segments, list):
                segments = []
            cleaned_segments = [s.strip() for s in segments if isinstance(s, str) and s.strip()]

            # Ensure junctions is a list
            if not isinstance(junctions, list):
                junctions = []
            cleaned_junctions = [j.strip() for j in junctions if isinstance(j, str) and j.strip()]

            all_connections.append({
                "Page": page_idx,
                "From_Connector": from_conn,
                "To_Connector": to_conn,
                "Wire_ID": wire_id,
                "Wire_Gauge": wire_gauge,
                "Wire_Color": wire_color,
                "Core_Count": core_count,
                "Total_Segments": len(cleaned_segments),
                "Segment_Lengths": cleaned_segments,
                "Junction_Points": cleaned_junctions,
                "Notes": notes
            })

            # Print summary for this connection
            print(f"      [{conn_idx}] {from_conn} → {to_conn}")
            print(f"          Wire: {wire_id or 'N/A'}, Gauge: {wire_gauge or 'N/A'}, Color: {wire_color or 'N/A'}")
            print(f"          Segments: {len(cleaned_segments)} → {cleaned_segments if cleaned_segments else 'No dimensions'}")
            if cleaned_junctions:
                print(f"          Junctions: {', '.join(cleaned_junctions)}")

    # 3) Summary
    print(f"\n{'='*60}")
    print(f"EXTRACTION SUMMARY")
    print(f"{'='*60}")
    print(f"Total Pages Processed: {len(temp_images)}")
    print(f"Tables Found: {len(tables_found)}")
    for t in tables_found:
        print(f"  - Page {t['page']}: {t['location']}")
    print(f"Total Connections Extracted: {len(all_connections)}")

    if not all_connections:
        print("\n⚠ No connections extracted. No Excel file created.")
        return

    # 4) Create detailed Excel output
    print(f"\n📝 Creating Excel output...")

    # Flatten segment lengths and junction points into columns
    max_segments = max((row.get("Total_Segments", 0) for row in all_connections), default=0)
    max_junctions = max((len(row.get("Junction_Points", [])) for row in all_connections), default=0)

    flat_rows = []
    for row in all_connections:
        base = {
            "Page": row["Page"],
            "From_Connector": row["From_Connector"],
            "To_Connector": row["To_Connector"],
            "Wire_ID": row["Wire_ID"],
            "Wire_Gauge": row["Wire_Gauge"],
            "Wire_Color": row["Wire_Color"],
            "Core_Count": row["Core_Count"],
            "Total_Segments": row["Total_Segments"],
        }
        
        # Add segment columns
        segs = row.get("Segment_Lengths", [])
        for idx in range(max_segments):
            col_name = f"Segment_{idx + 1}"
            base[col_name] = segs[idx] if idx < len(segs) else None
        
        # Add junction columns
        juncs = row.get("Junction_Points", [])
        for idx in range(max_junctions):
            col_name = f"Junction_{idx + 1}"
            base[col_name] = juncs[idx] if idx < len(juncs) else None
        
        base["Notes"] = row.get("Notes")
        flat_rows.append(base)

    df_conn = pd.DataFrame(flat_rows)

    # Create output filename
    output_filename = f"{Path(file_path).stem}_CONNECTOR_MAPPING_SEGMENTS.xlsx"

    with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
        df_conn.to_excel(writer, sheet_name="Connector_Mappings", index=False)
        
        # Add a summary sheet
        summary_data = {
            "Metric": [
                "Total Pages",
                "Tables Found",
                "Total Connections",
                "Unique From Connectors",
                "Unique To Connectors",
                "Connections with Segments",
                "Max Segments per Connection"
            ],
            "Value": [
                len(temp_images),
                len(tables_found),
                len(all_connections),
                df_conn["From_Connector"].nunique(),
                df_conn["To_Connector"].nunique(),
                (df_conn["Total_Segments"] > 0).sum(),
                max_segments
            ]
        }
        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name="Summary", index=False)

    print(f"\n✅ SUCCESS! Excel file created: {output_filename}")
    print(f"   - Sheet 1: Connector_Mappings ({len(flat_rows)} rows)")
    print(f"   - Sheet 2: Summary")

    # 5) Cleanup temp images
    if ext == ".pdf":
        for t in temp_images:
            Path(t).unlink(missing_ok=True)
        print(f"\n🧹 Cleaned up {len(temp_images)} temporary image files")


# ==========================
# ENTRY POINT
# ==========================

if __name__ == "__main__":
    print("\n" + "="*60)
    print("ENHANCED CONNECTION EXTRACTOR")
    print("Table-Focused Connector Mapping with Segmented Paths")
    print("="*60)
    
    # Diagnostics
    print("\n🔧 System Check:")
    print(f"   → Python: {Path.cwd()}")
    
    # Check for pdf2image
    try:
        from pdf2image import convert_from_path
        print("   ✓ pdf2image installed")
    except ImportError:
        print("   ❌ pdf2image NOT installed - run: pip install pdf2image")
    
    # Check for poppler (on some systems)
    import subprocess
    import platform
    try:
        if platform.system() == "Windows":
            result = subprocess.run(["where", "pdftoppm"], capture_output=True, text=True)
        else:
            result = subprocess.run(["which", "pdftoppm"], capture_output=True, text=True)
        
        if result.returncode == 0:
            print("   ✓ Poppler utilities found")
        else:
            print("   ⚠ Poppler utilities not found in PATH")
            print("     Install: https://github.com/oschwartz10612/poppler-windows (Windows)")
            print("     Or: sudo apt-get install poppler-utils (Linux)")
            print("     Or: brew install poppler (Mac)")
    except Exception:
        print("   ⚠ Could not check for poppler")
    
    print("\nReady to process files.")
    print("Call: process_file_enhanced('your_file.pdf')")
    print("\nExample:")
    print("  process_file_enhanced('Sanitized sheet 1 (var 2).pdf')")

Enhanced Connection Extractor Ready (Table-Focused with Segment Mapping).

ENHANCED CONNECTION EXTRACTOR
Table-Focused Connector Mapping with Segmented Paths

🔧 System Check:
   → Python: /home/jovyan/Data Extraction
   ✓ pdf2image installed
   ✓ Poppler utilities found

Ready to process files.
Call: process_file_enhanced('your_file.pdf')

Example:
  process_file_enhanced('Sanitized sheet 1 (var 2).pdf')


In [6]:
process_file_enhanced("Master test.png")


Processing Engineering Drawing: Master test.png

🖼️ Processing image file directly...
   ✓ Image file found

────────────────────────────────────────────────────────────
📊 Analyzing Page 1/1
────────────────────────────────────────────────────────────
   -> Analyzing: Master test.png...
   ✓ Table found: Multiple tables distributed around the drawing (C1, P1, C2, C3, and bottom-left).
   → Extracted 14 connections
      [1] C1 → P1
          Wire: Wire1.Red, Gauge: 20 AWG, Color: Red
          Segments: 4 → ['1.5 in', '7 ft', '3 ft', '7 ft']
          Junctions: Junction1, Junction2, Junction3
      [2] C1 → P1
          Wire: Wire2.Orange, Gauge: 20 AWG, Color: Orange
          Segments: 4 → ['1.5 in', '7 ft', '3 ft', '7 ft']
          Junctions: Junction1, Junction2, Junction3
      [3] C1 → P1
          Wire: Wire3.Green, Gauge: 20 AWG, Color: Green
          Segments: 4 → ['1.5 in', '7 ft', '3 ft', '7 ft']
          Junctions: Junction1, Junction2, Junction3
      [4] C1 → P1
    